# 3.1 MCQ Shift Golden Answer Location

## Pull in the data direct from HF

In [2]:
import pandas as pd

# Load the dataset from the provided URL
url = "https://huggingface.co/datasets/rabell/SysEngBench/resolve/main/test.csv"
df = pd.read_csv(url)

In [3]:
df

,Question ID,Tags,INCOSE Handbook Category,question,choiceA,choiceB,choiceC,choiceD,answer,label,Justification
0,1,Introduction to risk,INCOSEHandbook/Systems Engineering Overview/Sy...,What best describes the concept of uncertainty...,The process of systematically improving and op...,The condition where the outcomes of system fun...,A method for analyzing the costs and benefits ...,The act of integrating different system compon...,B,1,Uncertainty in systems engineering refers to t...
1,2,Introduction to risk,INCOSEHandbook/Systems Engineering Overview/Sy...,How is risk defined in systems engineering?,The guaranteed outcome of a system's failure.,The process of optimizing a system to avoid an...,The potential for loss or an undesirable outco...,The act of integrating different system compon...,C,2,Risk in systems engineering is conceptualized ...
2,3,Introduction to risk,INCOSEHandbook/Systems Engineering Overview/Sy...,Which of the following best describes the two ...,Uncertainty due to incomplete testing and unce...,Uncertainty due to a lack of knowledge that ca...,Uncertainty due to software errors and uncerta...,Uncertainty due to environmental factors and u...,B,1,Uncertainty in systems engineering can often b...
3,4,Introduction to risk,INCOSEHandbook/Systems Engineering Overview/Sy...,What is the primary goal of systems integratio...,To reduce the overall cost of the system.,To enhance the individual performance of each ...,To ensure that system components function toge...,To document the system's requirements and spec...,C,2,The primary goal of systems integration in sys...
4,5,Introduction to risk,INCOSEHandbook/Systems Engineering Overview/Sy...,What best describes the concept of reliability...,The ability of a system to be produced within ...,The capacity of a system to perform its requir...,The process of ensuring a system is resistant ...,The efficiency with which a system uses resour...,B,1,Reliability in systems engineering refers to t...
...,...,...,...,...,...,...,...,...,...,...,...
1139,1140,Design Standards,INCOSEHandbook/Specialty Engineering Activitie...,Which test within MIL-STD-461 evaluates equipm...,RE101,RE102,RS103,CE102,B,1,RE102 assesses radiated electric field emissio...
1140,1141,Design Standards,INCOSEHandbook/Specialty Engineering Activitie...,Which test within MIL-STD-461 evaluates equipm...,RE102,RE103,CE106,CS103,B,1,RE103 measures spurious and harmonic emissions...
1141,1142,Design Standards,INCOSEHandbook/Specialty Engineering Activitie...,Which test within MIL-STD-461 evaluates equipm...,CE102,CS104,CS118,RS101,D,3,RS101 assesses equipment susceptibility to rad...
1142,1143,Design Standards,INCOSEHandbook/Specialty Engineering Activitie...,Which test within MIL-STD-461 evaluates equipm...,RS103,RS101,RE102,CS103,A,0,RS103 evaluates susceptibility to radiated ele...


## Shifting to target position

In [8]:
import pandas as pd
from pathlib import Path

def _rotate(lst, k):
    k = k % 4
    return lst[-k:] + lst[:-k]

# Align each row so the correct (golden) answer moves to a target letter (A/B/C/D)
def align_choices_to_target(row, target_letter='A'):
    letters = ['A', 'B', 'C', 'D']
    choices_cols = ['choiceA', 'choiceB', 'choiceC', 'choiceD']

    correct_letter = str(row['answer']).strip().upper()
    if correct_letter not in letters:
        raise ValueError(f"Unexpected answer value: {row['answer']} (expected one of A/B/C/D)")

    current_idx = letters.index(correct_letter)
    target_idx = letters.index(target_letter)
    shift_by = (target_idx - current_idx) % 4

    choice_texts = [row[c] for c in choices_cols]
    shifted = _rotate(choice_texts, shift_by)

    return {
        'choiceA': shifted[0],
        'choiceB': shifted[1],
        'choiceC': shifted[2],
        'choiceD': shifted[3],
        'answer': target_letter,  # force golden to target
        'label': target_idx       # 0=A, 1=B, 2=C, 3=D
    }

def create_variants_by_golden(df):
    variants = {}
    for target in ['A', 'B', 'C', 'D']:
        dfx = df.copy()
        aligned_rows = dfx.apply(lambda r: align_choices_to_target(r, target), axis=1)
        dfx[['choiceA', 'choiceB', 'choiceC', 'choiceD', 'answer', 'label']] = pd.DataFrame(aligned_rows.tolist())
        variants[target] = dfx
    return variants

# Resolve project root relative to this notebook
try:
    notebook_dir = Path(__file__).resolve().parent  # __file__ is not set in notebooks
except NameError:
    notebook_dir = Path.cwd()

project_root = notebook_dir.parent.parent  # .../dissertation
output_dir = project_root / "src" / "phase3_variants"
output_dir.mkdir(parents=True, exist_ok=True)

# Build the four variants and save with relative paths
variants = create_variants_by_golden(df)
name_map = {
    'A': "sysengbench_a.csv",
    'B': "sysengbench_b.csv",
    'C': "sysengbench_c.csv",
    'D': "sysengbench_d.csv",
}

output_paths = {}
for letter, dfx in variants.items():
    out_path = output_dir / name_map[letter]
    dfx.to_csv(out_path, index=False)
    output_paths[letter] = str(out_path.relative_to(project_root))

output_paths  # shows relative locations from repo root

{'A': 'src\\phase3_variants\\sysengbench_a.csv',
 'B': 'src\\phase3_variants\\sysengbench_b.csv',
 'C': 'src\\phase3_variants\\sysengbench_c.csv',
 'D': 'src\\phase3_variants\\sysengbench_d.csv'}

## Checking that the shift worked

In [ ]:

from pathlib import Path
import pandas as pd
from IPython.display import display

# Preview: show the first row from each of the four variants (A/B/C/D)
letters = ['A', 'B', 'C', 'D']

def preview_first_rows():
    try:
        # Prefer reading what was written to disk
        for letter in letters:
            csv_name = name_map[letter]  # defined above
            csv_path = output_dir / csv_name  # defined above
            df_preview = pd.read_csv(csv_path, nrows=1)
            print(f"== {letter} variant -> {csv_name} ==")
            display(df_preview)
    except Exception as e:
        # Fallback: preview from in-memory DataFrames if available
        if 'variants' in globals():
            for letter in letters:
                print(f"== {letter} variant (in-memory) ==")
                display(variants[letter].head(1))
        else:
            print(f"Preview failed: {e}")

preview_first_rows()


== A variant -> sysengbench_a.csv ==


,Question ID,Tags,INCOSE Handbook Category,question,choiceA,choiceB,choiceC,choiceD,answer,label,Justification
0,1,Introduction to risk,INCOSEHandbook/Systems Engineering Overview/Sy...,What best describes the concept of uncertainty...,The condition where the outcomes of system fun...,A method for analyzing the costs and benefits ...,The act of integrating different system compon...,The process of systematically improving and op...,A,0,Uncertainty in systems engineering refers to t...


== B variant -> sysengbench_b.csv ==


,Question ID,Tags,INCOSE Handbook Category,question,choiceA,choiceB,choiceC,choiceD,answer,label,Justification
0,1,Introduction to risk,INCOSEHandbook/Systems Engineering Overview/Sy...,What best describes the concept of uncertainty...,The process of systematically improving and op...,The condition where the outcomes of system fun...,A method for analyzing the costs and benefits ...,The act of integrating different system compon...,B,1,Uncertainty in systems engineering refers to t...


== C variant -> sysengbench_c.csv ==


,Question ID,Tags,INCOSE Handbook Category,question,choiceA,choiceB,choiceC,choiceD,answer,label,Justification
0,1,Introduction to risk,INCOSEHandbook/Systems Engineering Overview/Sy...,What best describes the concept of uncertainty...,The act of integrating different system compon...,The process of systematically improving and op...,The condition where the outcomes of system fun...,A method for analyzing the costs and benefits ...,C,2,Uncertainty in systems engineering refers to t...


== D variant -> sysengbench_d.csv ==


,Question ID,Tags,INCOSE Handbook Category,question,choiceA,choiceB,choiceC,choiceD,answer,label,Justification
0,1,Introduction to risk,INCOSEHandbook/Systems Engineering Overview/Sy...,What best describes the concept of uncertainty...,A method for analyzing the costs and benefits ...,The act of integrating different system compon...,The process of systematically improving and op...,The condition where the outcomes of system fun...,D,3,Uncertainty in systems engineering refers to t...
